<a href="https://colab.research.google.com/github/jomanakhatib/Projects-/blob/main/Milestone_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milesstone 2
Jomana Mohamed Magdy

58-4193

T10

Malak Osama Elmanhy

58-3344

T10

#Password Encryption

# Data Loading

In [18]:

import hashlib
import os
import time
import binascii
from typing import Tuple

#VIGENERE CIPHER

In [19]:
 #Key Preperation
def vigenere_prepare_key(password: str, key: str) -> str:
    key = key.replace(" ", "")
    if len(key) == 0:
        raise ValueError("Key cannot be empty")
    # repeat or truncate key to match password length
    repeated = (key * ((len(password) // len(key)) + 1))[:len(password)]
    return repeated
 # Key Encryption
def vigenere_encrypt(plaintext: str, key: str) -> str:
    plaintext = plaintext  # preserve case (we'll operate on all bytes)
    key_seq = vigenere_prepare_key(plaintext, key)
    ciphertext = []
    for p_char, k_char in zip(plaintext, key_seq):
        # Support all printable ASCII by shifting over 256 space using bytes:
        p = ord(p_char)
        k = ord(k_char)
        c = (p + k) % 256
        ciphertext.append(chr(c))
    return "".join(ciphertext)
 # Key Decryption
def vigenere_decrypt(ciphertext: str, key: str) -> str:
    key_seq = vigenere_prepare_key(ciphertext, key)
    plaintext = []
    for c_char, k_char in zip(ciphertext, key_seq):
        c = ord(c_char)
        k = ord(k_char)
        p = (c - k) % 256
        plaintext.append(chr(p))
    return "".join(plaintext)

# PLAYFAIR CIPHER

In [20]:
def playfair_create_matrix(key: str, alphabet: str = "ABCDEFGHIKLMNOPQRSTUVWXYZ") -> list:
    # default merges I/J (classic 5x5)
    # Normalize key: uppercase + remove non-letters + replace J with I
    key = "".join([c.upper() for c in key if c.isalpha()])
    key = key.replace("J", "I")
    seen = []
    for ch in key + alphabet:
        if ch not in seen:
            seen.append(ch)
    # convert to 5x5 matrix
    matrix = [seen[i*5:(i+1)*5] for i in range(5)]
    return matrix

def playfair_find(matrix, ch):
    for r in range(5):
        for c in range(5):
            if matrix[r][c] == ch:
                return r, c
    raise ValueError(f"{ch} not in matrix")

def playfair_prepare_text(plaintext: str) -> list:
    # remove non-letters, uppercase, merge J->I
    s = "".join([c.upper() for c in plaintext if c.isalpha()]).replace("J", "I")
    pairs = []
    i = 0
    while i < len(s):
        a = s[i]
        b = s[i+1] if i+1 < len(s) else None
        if b is None:
            pairs.append((a, 'X'))
            i += 1
        elif a == b:
            # insert filler (X)
            pairs.append((a, 'X'))
            i += 1
        else:
            pairs.append((a, b))
            i += 2
    return pairs

def playfair_encrypt(plaintext: str, key: str) -> str:
    matrix = playfair_create_matrix(key)
    pairs = playfair_prepare_text(plaintext)
    cipher = []
    for a,b in pairs:
        ra, ca = playfair_find(matrix, a)
        rb, cb = playfair_find(matrix, b)
        if ra == rb:
            cipher.append(matrix[ra][(ca + 1) % 5])
            cipher.append(matrix[rb][(cb + 1) % 5])
        elif ca == cb:
            cipher.append(matrix[(ra + 1) % 5][ca])
            cipher.append(matrix[(rb + 1) % 5][cb])
        else:
            cipher.append(matrix[ra][cb])
            cipher.append(matrix[rb][ca])
    return "".join(cipher)

def playfair_decrypt(ciphertext: str, key: str) -> str:
    matrix = playfair_create_matrix(key)
    s = "".join([c.upper() for c in ciphertext if c.isalpha()])
    if len(s) % 2 != 0:
        raise ValueError("Ciphertext length must be even for Playfair")
    plain = []
    for i in range(0, len(s), 2):
        a, b = s[i], s[i+1]
        ra, ca = playfair_find(matrix, a)
        rb, cb = playfair_find(matrix, b)
        if ra == rb:
            plain.append(matrix[ra][(ca - 1) % 5])
            plain.append(matrix[rb][(cb - 1) % 5])
        elif ca == cb:
            plain.append(matrix[(ra - 1) % 5][ca])
            plain.append(matrix[(rb - 1) % 5][cb])
        else:
            plain.append(matrix[ra][cb])
            plain.append(matrix[rb][ca])
    return "".join(plain)

#  PBKDF2 (3rd method)

In [21]:
def pbkdf2_hash(password: str, iterations: int = 200_000, dklen: int = 32, salt: bytes = None) -> Tuple[str, str, int]:
    """
    Return (salt_hex, derived_key_hex, iterations)
    - iterations default: 200k (example; tune based on performance)
    """
    if salt is None:
        salt = os.urandom(16)
    dk = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, iterations, dklen=dklen)
    return binascii.hexlify(salt).decode(), binascii.hexlify(dk).decode(), iterations

def pbkdf2_verify(password: str, salt_hex: str, dk_hex: str, iterations: int) -> bool:
    salt = binascii.unhexlify(salt_hex)
    expected = binascii.unhexlify(dk_hex)
    test = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, iterations, dklen=len(expected))
    return test == expected

# Evaluating the 3 cryptographic techniques

In [24]:
def demo_encrypt_all(password: str, key_vig: str, key_play: str):
    print("Original password:", password)
    print("\n--- Vigenere ---")
    t0 = time.time()
    vig_ct = vigenere_encrypt(password, key_vig)
    t1 = time.time()
    print("Ciphertext (hex):", binascii.hexlify(vig_ct.encode()).decode())
    print("Time (ms):", round((t1 - t0) * 1000, 3))
    # decryption check
    assert vigenere_decrypt(vig_ct, key_vig) == password

    print("\n--- Playfair ---")
    t0 = time.time()
    play_ct = playfair_encrypt(password, key_play)
    t1 = time.time()
    print("Ciphertext:", play_ct)
    print("Time (ms):", round((t1 - t0) * 1000, 3))
    # decryption check
    try:
        play_dec = playfair_decrypt(play_ct, key_play)
        print("Decrypted (Playfair):", play_dec)
    except Exception as e:
        print("Playfair decryption error (expected for odd-length/format issues):", e)

    print("\n--- PBKDF2 (hash) ---")
    t0 = time.time()
    salt_hex, dk_hex, iterations = pbkdf2_hash(password)
    t1 = time.time()
    print("Salt (hex):", salt_hex)
    print("Derived key (hex):", dk_hex)
    print("Iterations:", iterations)
    print("Time (ms):", round((t1 - t0) * 1000, 3))
    # verify
    ok = pbkdf2_verify(password, salt_hex, dk_hex, iterations)
    print("PBKDF2 verification OK:", ok)

# Testing

In [23]:
if __name__ == "__main__":
    # Example — change to test other values
    sample_password = "My$ecretP@ssw0rd!"
    vig_key = "lemon"           # Vigenere key (example)
    play_key = "keyword"        # Playfair key (example)
    print("Running demo...\n")
    demo_encrypt_all(sample_password, vig_key, play_key)

Running demo...

Original password: My$ecretP@ssw0rd!

--- Vigenere ---
Ciphertext (hex): c2b9c39ec291c394c391c39ec38ac3a1c2bfc2aec39fc398c3a4c29fc3a0c390c286
Time (ms): 0.021

--- Playfair ---
Ciphertext: PKODDKVMQZQODA
Time (ms): 0.045
Decrypted (Playfair): MYECRETPSXSWRD

--- PBKDF2 (hash) ---
Salt (hex): 432926e0c0971c2780f1783e7f08b5fe
Derived key (hex): f3e4f9146de156675cadaa7d982c7c2e7013a797ad15bd3f0261d08331d81d17
Iterations: 200000
Time (ms): 163.749
PBKDF2 verification OK: True
